# Synthetic Groundedness Dataset Preparation - Domain-specific Set of Bank Retail Customer Queries - Construct Independent Sample for Cross-encoder Training

Information on the dataset can be found here: https://huggingface.co/datasets/PolyAI/banking77
Dataset was downloaded from here: https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data
Training dataset was used. 

In [ ]:
# Install required packages (if needed)
#!pip install openai requests pandas

# Import required libraries
import os
from openai import OpenAI, OpenAIError, RateLimitError
import requests
import pandas as pd
import random
import time
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from .env file
load_dotenv(dotenv_path=".env", override=True)

# Configure API client
OpenAI.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OpenAI.api_key)

# Define which models to use for context generation, grounded responses, and hallucinated responses.
MODEL_GENERATE_CONTEXT = "gpt-4o-mini" 
MODEL_GROUNDED = "gpt-4o"         
MODEL_HALLUCINATED = "gpt-3.5-turbo-0125"

# File paths for data flow
SAMPLED_QUERIES_PATH = "../data/domain_specific/interim_data/banking_77_queries_1k_training.csv"
CONTEXT_OUTPUT_PATH = "../data/domain_specific/interim_data/banking_77_queries_1k_with_context.csv"
INTERIM_OUTPUT_PATH = "../data/domain_specific/interim_data/banking_77_1k_triplet_set.csv"

In [2]:
# Load the datasets
main_df = pd.read_csv("../data/domain_specific/raw_data/banking_77_train_set.csv") # Downloaded from Banking 77 website
sampled_df = pd.read_csv("../data/domain_specific/sampled_data/initial_sampled_banking_77_queries.csv")  # Initial set of queries sampled that will now be used as the holdout set for evaluation

# Remove sampled queries from main_df
filtered_df = main_df[~main_df['text'].isin(sampled_df['query'])]

# Confirm removal
removed = main_df[main_df['text'].isin(sampled_df['query'])]

print(f"Number of queries removed: {len(removed)}")
print("Sample of removed queries:")
print(removed[['text', 'category']].sample(n=min(5, len(removed))))

# Check these should not be in the filtered_df
still_present = filtered_df[filtered_df['text'].isin(sampled_df['query'])]
if still_present.empty:
    print("Confirmation: No sampled queries remain in the filtered dataset.")
else:
    print(" Warning: Some sampled queries are still present in the filtered dataset:")
    print(still_present[['text', 'category']])

# Save filtered dataset
filtered_df.to_csv("../data/domain_specific/sampled_data/banking_77_train_set_filtered.csv", index=False)
print("Filtered dataset saved to banking_77_train_set_filtered.csv")


Number of queries removed: 385
Sample of removed queries:
                                                   text  \
2753  Why is it taking so long for a transfer to com...   
7634             How can I deposit money to my account?   
3343  I deposited a cheque but the cash hasn't arriv...   
9815  If I don't have all my documents can they stil...   
1383  With this app, will I be able to exchange curr...   

                                              category  
2753                transfer_not_received_by_recipient  
7634                             transfer_into_account  
3343  balance_not_updated_after_cheque_or_cash_deposit  
9815                                verify_my_identity  
1383                                  exchange_via_app  
Confirmation: No sampled queries remain in the filtered dataset.
Filtered dataset saved to banking_77_train_set_filtered.csv


### Sampling Customer Queries for Crossencoder Training

In [3]:
# Load initial customer query dataset
banking_df = pd.read_csv("../data/domain_specific/sampled_data/banking_77_train_set_filtered.csv")
banking_df = banking_df.dropna(subset=["text"])    # Ensure no missing queries
banking_df = banking_df.rename(columns={"text": "query"})  # Rename 'text' to 'query' for consistency

# Proportional sampling: maintain the original category distribution
# Sample 1000 queries proportionally from the full banking_df
proportional_sample_df = banking_df.sample(
    n=1000,
    weights=banking_df['category'].map(banking_df['category'].value_counts()),
    random_state=42
)

# Check the resulting distribution
prop_category_counts = proportional_sample_df['category'].value_counts()
prop_category_percent = (prop_category_counts / len(proportional_sample_df) * 100).round(2)

# Calculate full dataset category distribution
full_category_counts = banking_df['category'].value_counts()
full_category_percent = (full_category_counts / len(banking_df) * 100).round(2)

# Combine for comparison
prop_comparison_df = pd.DataFrame({
    'full_count': full_category_counts,
    'full_percent': full_category_percent,
    'sampled_count': prop_category_counts,
    'sampled_percent': prop_category_percent
}).fillna(0).astype({'full_count': int, 'sampled_count': int}).reset_index().rename(columns={'index': 'category'})

print(prop_comparison_df)
len(proportional_sample_df)
proportional_sample_df.to_csv(SAMPLED_QUERIES_PATH, index=False)


                                   category  full_count  full_percent  \
0                     Refund_not_showing_up         157          1.63   
1                          activate_my_card         154          1.60   
2                                 age_limit         105          1.09   
3                   apple_pay_or_google_pay         121          1.26   
4                               atm_support          82          0.85   
..                                      ...         ...           ...   
72                 virtual_card_not_working          36          0.37   
73                       visa_or_mastercard         130          1.35   
74                      why_verify_identity         116          1.21   
75            wrong_amount_of_cash_received         175          1.82   
76  wrong_exchange_rate_for_cash_withdrawal         158          1.64   

    sampled_count  sampled_percent  
0              20              2.0  
1              18              1.8  
2           

### Generate Synthetic FAQ Context for Each Query

In [ ]:
# This section generates realistic help content using GPT for each sampled query.
SAVE_INTERVAL = 40
MAX_RETRIES = 6
SLEEP_BETWEEN_REQUESTS = 1  # seconds

df_sampled = pd.read_csv(SAMPLED_QUERIES_PATH)

# Prompt template for generating help content
def generate_context_prompt(query, category):
    return f"""You are writing content for the Help or FAQ section of a retail bank's website. 

Write a synthetic help page excerpt (150–250 words) that would support answering the following customer query, using appropriate terminology and informative tone typical of a real bank website. Can you use similar information as what you would find on the big four banks in Australia's websites and make sure they are banking products you would find in Australia. 

Category: {category}
Query: "{query}"
"""

# Load existing file or initialise fresh run
df_sampled["context"] = None
if Path(CONTEXT_OUTPUT_PATH).exists():
    df_existing = pd.read_csv(CONTEXT_OUTPUT_PATH)
    df_sampled.loc[df_existing.index, "context"] = df_existing["context"]
    print(f"Resuming from row {df_existing['context'].last_valid_index() + 1}")

# Generate missing contexts using GPT
for i, row in df_sampled.iterrows():
    if pd.notnull(row["context"]):
        continue

    prompt = generate_context_prompt(row["query"], row["category"])
    retries = 0
    context = None

    while retries < MAX_RETRIES:
        try:
            response = client.chat.completions.create(
                model=MODEL_GENERATE_CONTEXT,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7
            )
            context = response.choices[0].message.content.strip()
            break
        except RateLimitError:
            wait_time = 2 ** retries
            print(f"[Rate Limit] Row {i}: waiting {wait_time}s (retry {retries + 1})")
            time.sleep(wait_time)
            retries += 1
        except OpenAIError as e:
            print(f"[OpenAIError] Row {i}: {e}")
            break
        except Exception as e:
            print(f"[Unexpected Error] Row {i}: {e}")
            break

    df_sampled.at[i, "context"] = context if context else "ERROR"
    time.sleep(SLEEP_BETWEEN_REQUESTS)

    if i % SAVE_INTERVAL == 0 or i == len(df_sampled) - 1:
        df_sampled.to_csv(CONTEXT_OUTPUT_PATH, index=False)
        print(f"Progress saved up to row {i}")

print("Context generation complete.")


#### Generate Grounded and Hallucinated Responses, and save final Challenger Set

In [5]:
# This section loads the context-enriched queries to generate grounded and hallucinated responses.
# Load previously generated synthetic context data
df = pd.read_csv(CONTEXT_OUTPUT_PATH)
df["response_grounded"] = None
df["response_ungrounded"] = None

# Define prompt templates
# These functions generate the instructions for GPT to create grounded or hallucinated answers.
def prompt_grounded(query, context):
    return f"""You are a customer support assistant. Using ONLY the information provided below, write a helpful and accurate answer to the customer query.

DO NOT include any facts, assumptions, or language that are not directly supported by the context.

Context:
{context}

Query:
{query}
"""

def prompt_ungrounded(query, context):
    return f"""You are a customer support assistant. Write a plausible-sounding response to the customer's query, but include subtle hallucinations.

- Change numbers (e.g., fees, timeframes)
- Make up a service/product
- Assume a specific type of card or account
- Repeat the first part of the query in the response before shifting to unrelated details

Do NOT reuse the content directly from context. Make the hallucination subtle but detectable.

Query:
{query}

Context (do NOT copy from this):
{context}
"""

In [ ]:
# API Wrapper for Completion Calls
# Handles GPT API calls with retry logic in case of errors or rate limits.
def generate_response(prompt, model, temperature=0.0, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Retry {attempt + 1} failed: {e}")
            time.sleep(2)
    return None

# Generate and Save Responses - generates grounded and hallucinated answers for each query and saves output incrementally.
for i, row in df.iterrows():
    if pd.notnull(row["response_grounded"]) and pd.notnull(row["response_ungrounded"]):
        continue

    query = row["query"]
    context = row["context"]

    grounded = generate_response(prompt_grounded(query, context), MODEL_GROUNDED, temperature=0.0)
    time.sleep(1)
    ungrounded = generate_response(prompt_ungrounded(query, context), MODEL_HALLUCINATED, temperature=0.8)
    time.sleep(1)

    df.at[i, "response_grounded"] = grounded
    df.at[i, "response_ungrounded"] = ungrounded

    if i % 20 == 0:
        df.to_csv(INTERIM_OUTPUT_PATH, index=False)
        print(f"Progress saved at index {i}")

# Final save
df.to_csv(INTERIM_OUTPUT_PATH, index=False)
print(f"Final data saved to {INTERIM_OUTPUT_PATH}")

In [ ]:
# Convert dataset to format required for evaluators
# Load the challenger dataset
df = pd.read_csv("../data/domain_specific/interim_data/banking_77_1k_triplet_set.csv")

# Create PASS rows
df_pass = df[["query", "context", "response_grounded"]].copy()
df_pass["label"] = "PASS"
df_pass.rename(columns={"response_grounded": "response"}, inplace=True)

# Create FAIL rows
df_fail = df[["query", "context", "response_ungrounded"]].copy()
df_fail["label"] = "FAIL"
df_fail.rename(columns={"response_ungrounded": "response"}, inplace=True)

# Combine them
df_long = pd.concat([df_pass, df_fail], ignore_index=True)

# Save to CSV
df_long.to_csv("../data/domain_specific/banking_77_1k_triplet_set_long.csv", index=False)

In [ ]:
df

,query,category,context,response_grounded,response_ungrounded
0,How do I edit my details?,edit_personal_details,### How to Edit Your Personal Details\n\nKeepi...,"To edit your personal details, please follow t...","To edit your details, please log into your acc..."
1,is there a charge on withdrawals?,cash_withdrawal_charge,"### Cash Withdrawal Charges\n\nAt [Bank Name],...",When you withdraw cash from an ATM within our ...,"Yes, there is a charge on withdrawals, but it ..."
2,Why am I seeing a fee for transferring money?,transfer_fee_charged,**Understanding Transfer Fees**\n\nAt [Bank Na...,The fee you are seeing for transferring money ...,We appreciate your inquiry regarding the fee y...
3,"I keep trying to make a payment, but it doesn'...",declined_card_payment,### Troubleshooting Declined Card Payments\n\n...,I'm sorry to hear you're having trouble with y...,I understand that you're having trouble making...
4,Do you have a children account available?,age_limit,**Children's Accounts: An Overview**\n\nAt [Yo...,"Yes, we offer a Children’s Account designed fo...","Yes, we do have a children's account available..."
...,...,...,...,...,...
995,What happened to the transfer I did?,balance_not_updated_after_bank_transfer,**Understanding Your Bank Transfer Status**\n\...,If your transfer hasn't updated in your accoun...,What happened to the transfer you did? Sometim...
996,"I cannot find my phone, what should I do?",lost_or_stolen_phone,**Help & FAQ: What to Do If You Cannot Find Yo...,"If you cannot find your phone, please follow t...",I understand your concern about not being able...
997,"When I withdrew my money, why was there an ext...",cash_withdrawal_charge,**Understanding Cash Withdrawal Charges**\n\nA...,There could be several reasons for the extra c...,"When you withdrew your money, the extra charge..."
998,I wish to be able to top up with cash.,top_up_by_cash_or_cheque,### Top Up Your Account with Cash or Cheque\n\...,You can top up your account with cash by visit...,I understand that you wish to be able to top u...
